<a href="https://colab.research.google.com/github/Ansh-Mistry/SLM/blob/main/SLM_CustomerSupport.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected! Go to Runtime > Change runtime type > select GPU")



CUDA available: True
GPU name: Tesla T4


In [ ]:
!pip install torch torchvision torchaudio --upgrade
!pip install transformers datasets tiktoken tqdm

In [ ]:
block_size = 128  # 👈 must stay consistent everywhere


In [ ]:
from datasets import load_dataset
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
print(ds)
print(ds["train"][0])


DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [ ]:
from datasets import load_dataset

# Load the Bitext customer support dataset
ds = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

# Check dataset structure
print(ds)

# Look at one example
print(ds["train"][0])


DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [ ]:
import pandas as pd

# Convert dataset to pandas for easier handling
df = pd.DataFrame(ds["train"])

# Let's assume dataset has 'instruction' (question) and 'response' (answer)
# If column names differ, replace accordingly
df["text"] = df["instruction"] + "\nAnswer: " + df["response"]

# Save as a CSV
df.to_csv("customer_support.csv", index=False)

print("Saved dataset with", len(df), "examples")
print(df.head(5))



Saved dataset with 26872 examples
   flags                                        instruction category  \
0      B   question about cancelling order {{Order Number}}    ORDER   
1    BQZ  i have a question about cancelling oorder {{Or...    ORDER   
2   BLQZ    i need help cancelling puchase {{Order Number}}    ORDER   
3     BL         I need to cancel purchase {{Order Number}}    ORDER   
4  BCELN  I cannot afford this order, cancel purchase {{...    ORDER   

         intent                                           response  \
0  cancel_order  I've understood you have a question regarding ...   
1  cancel_order  I've been informed that you have a question ab...   
2  cancel_order  I can sense that you're seeking assistance wit...   
3  cancel_order  I understood that you need assistance with can...   
4  cancel_order  I'm sensitive to the fact that you're facing f...   

                                                text  
0  question about cancelling order {{Order Number...  
1 

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
import numpy as np
import torch

# Load tokenizer (you can swap this with any model tokenizer, e.g. "gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Ensure tokenizer has a padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Reload processed CSV
df = pd.read_csv("customer_support.csv")

# Wrap into Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["instruction","response","text"], num_proc=2)

# Convert to torch tensors
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Split into train/val
split = tokenized.train_test_split(test_size=0.1, seed=42)
train_data, val_data = split["train"], split["test"]

# Save as binary files for your get_batch function
def save_bin(dataset, filename):
    arr = np.concatenate([dataset[i]["input_ids"].numpy() for i in range(len(dataset))])
    arr = arr.astype(np.uint16)
    arr.tofile(filename)

save_bin(train_data, "train.bin")
save_bin(val_data, "val.bin")

print("Saved train.bin and val.bin ✅")


Map (num_proc=2):   0%|          | 0/26872 [00:00<?, ? examples/s]

Saved train.bin and val.bin ✅


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------
# GPT Configuration
# ----------------------------
class GPTConfig:
    def __init__(self, block_size, vocab_size, n_layer, n_head, n_embd, dropout=0.1):
        self.block_size = block_size
        self.vocab_size = vocab_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout

# ----------------------------
# Multi-Head Attention
# ----------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1,1,config.block_size,config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

# ----------------------------
# Transformer Block
# ----------------------------
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# ----------------------------
# GPT Model
# ----------------------------
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Embedding(config.block_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.block_size = config.block_size

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.block_size

        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)

        return logits, loss


In [ ]:
import torch

# Example config (adjust if you want bigger model)
config = GPTConfig(
    block_size=block_size,
    vocab_size=50257,
    n_layer=4,
    n_head=4,
    n_embd=256,
)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GPT(config).to(device)
print(f"Model has {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

Model has 28.92M parameters


In [ ]:
from torch.amp import autocast, GradScaler
scaler = torch.amp.GradScaler(device="cuda")


In [ ]:
def get_batch(split):
    data = np.memmap(f"{split}.bin", dtype=np.uint16, mode="r")
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)


In [ ]:
# Training hyperparameters
max_iters = 10000 # Example: Number of training iterations
batch_size = 32
gradient_accumulation_steps = 4
learning_rate = 0.0003
use_amp = True # Set to False if you don't want to use mixed precision

In [ ]:
# Setup optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
from tqdm import tqdm # Import tqdm

best_val_loss = float("inf")  # ✅ define before training starts

print("🚀 Starting training...")
for step in tqdm(range(max_iters), desc="Training"):
    X, y = get_batch("train")
    optimizer.zero_grad(set_to_none=True)

    with autocast("cuda", dtype=torch.float16, enabled=use_amp):
        logits, loss = model(X, y)

    scaler.scale(loss).backward()

    if (step + 1) % gradient_accumulation_steps == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        if torch.isfinite(loss):
            scaler.step(optimizer)
            scaler.update()

    if step % 100 == 0:
        print(f"Step {step} | Loss: {loss.item():.4f}")

    # ✅ Validation + Save Best Model
    if step % 500 == 0 and step > 0:
        model.eval()
        with torch.no_grad():
            X_val, y_val = get_batch("val")
            with autocast("cuda", dtype=torch.float16, enabled=use_amp):
                _, val_loss = model(X_val, y_val)
        print(f"🔎 Validation | Step {step} | Val Loss: {val_loss.item():.4f}")

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            torch.save(model.state_dict(), "best_model.pt")
            print(f"💾 Saved new best model at step {step} | Val Loss: {best_val_loss:.4f}")

        model.train()

🚀 Starting training...


Training:   0%|          | 2/10000 [00:00<09:11, 18.13it/s]

Step 0 | Loss: 11.0604


Training:   1%|          | 101/10000 [00:06<12:44, 12.95it/s]

Step 100 | Loss: 5.4245


...

Training:  98%|█████████▊| 9801/10000 [11:17<00:15, 13.01it/s]

Step 9800 | Loss: 1.0318


Training:  99%|█████████▉| 9901/10000 [11:24<00:07, 12.91it/s]

Step 9900 | Loss: 1.0890


Training: 100%|██████████| 10000/10000 [11:31<00:00, 14.47it/s]


In [ ]:
torch.save(model.state_dict(), "gpt_model.pt")
torch.save(optimizer.state_dict(), "optimizer.pt")
print("✅ Model & optimizer saved.")


✅ Model & optimizer saved.


In [ ]:
# Same hyperparams as training
batch_size = 32
gradient_accumulation_steps = 4
learning_rate = 0.0003

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Initialize same model architecture
model = GPT(config).to(device)

# ✅ Load trained weights
model.load_state_dict(torch.load("gpt_model.pt", map_location=device))

# ✅ Setup optimizer again (must match training)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# ✅ Load optimizer state (so you can resume training exactly where you left off)
optimizer.load_state_dict(torch.load("optimizer.pt", map_location=device))

print("✅ Loaded trained model & optimizer from checkpoint")


✅ Loaded trained model & optimizer from checkpoint


In [ ]:
# STEP: Full validation pass (loss + perplexity)
import math
from torch.amp import autocast

model.eval()
eval_iters = 200 # you can raise to 500 for a tighter estimate
use_amp = True # keep same AMP setting you trained with
val_loss_accum = 0.0

with torch.no_grad():
    for _ in range(eval_iters):
        X_val, y_val = get_batch("val")
        with autocast("cuda", dtype=torch.float16, enabled=use_amp):
            _, loss = model(X_val, y_val)
            val_loss_accum += loss.item()


avg_val_loss = val_loss_accum / eval_iters
val_ppl = math.exp(avg_val_loss)

print(f"🔎 Validation — Avg Loss: {avg_val_loss:.4f} | Perplexity: {val_ppl:.3f}")

model.train() # switch back for any further training

🔎 Validation — Avg Loss: 0.8717 | Perplexity: 2.391


GPT(
  (tok_emb): Embedding(50257, 256)
  (pos_emb): Embedding(128, 256)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-3): 4 x Block(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=256, out_features=768, bias=True)
        (c_proj): Linear(in_features=256, out_features=256, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=256, out_features=50257, bias=False)
)

In [ ]:
# STEP 1: Inference setup — tokenizer + model load

import os, torch, tiktoken

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- tokenizer (same as training) ---
enc = tiktoken.get_encoding("gpt2")
def encode(text: str):
    return enc.encode_ordinary(text)
def decode(ids):
    return enc.decode(ids)

# --- rebuild the SAME model config you trained with ---
config = GPTConfig(
    block_size=128,      # <- same as training
    vocab_size=50257,    # GPT-2 tokenizer vocab
    n_layer=4,
    n_head=4,
    n_embd=256,
    dropout=0.1
)


# --- create model + load checkpoint ---
model = GPT(config).to(device)
ckpt_path = "best_model.pt" if os.path.exists("best_model.pt") else "gpt_model.pt"
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state)
model.eval()

print(f"✅ Loaded weights from {ckpt_path} on {device}")


✅ Loaded weights from best_model.pt on cuda


In [ ]:
import torch
import torch.nn.functional as F

def generate(model, tokenizer, prompt, max_new_tokens=50, temperature=0.8, top_k=50):
    """
    Generate text from the trained model.
    """
    model.eval()
    device = next(model.parameters()).device

    # Encode the prompt
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Forward pass
            logits, _ = model(input_ids)

            # Only use the last token's logits
            logits = logits[:, -1, :] / temperature

            # Top-k sampling
            if top_k is not None:
                v, ix = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("Inf")

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Append to sequence
            input_ids = torch.cat([input_ids, next_token], dim=1)

    # Decode back to text
    return tokenizer.decode(input_ids[0].tolist())


In [ ]:
from transformers import GPT2Tokenizer
import re

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

context_data = {
    "Refund Amount": "3-5",
    "Website URL": "https://example.com/support",
    "Customer Support Phone Number": "+1-200-333-4567",
    "Account Type": "Premium",
    "Order Number": "675846"
}

def clean_placeholders(text: str, context_data: dict):
    for key, val in context_data.items():
        text = text.replace(f"{{{{{key}}}}}", val)
    text = re.sub(r"\{\{.*?\}\}", "", text)  # drop unknowns
    return text.strip()

def guardrails(customer_input, agent_reply):
    if "discount" in customer_input.lower():
        return "Currently, we cannot guarantee any discount. Please check https://example.com/support for ongoing offers."
    if "cancel" in customer_input.lower():
        return "You can cancel your order within 3-5 business days by visiting https://example.com/support or calling +1-200-333-4567."
    return agent_reply

while True:
    customer_input = input("Customer: ")
    if customer_input.lower() == "exit":
        print("Exiting chat...")
        break

    full_prompt = f"Customer: {customer_input}\nAgent:"
    raw_response = generate(model, tokenizer, full_prompt, max_new_tokens=60, temperature=0.7, top_k=50)

    agent_reply = raw_response.split("Agent:")[-1].split("<|endoftext|>")[0].strip()
    agent_reply = clean_placeholders(agent_reply, context_data)
    agent_reply = guardrails(customer_input, agent_reply)

    print("Agent:", agent_reply)


Customer: how can i get discount
Agent: Currently, we cannot guarantee a 30% discount. Please check https://example.com/support for ongoing offers.
Customer: how can i talk with your manager
Agent: ** If you're unsure about the process of editing the allowed payment methods, please don't hesitate to let me know. We're here to help you make the payment process as possible.
Customer: how can i cancel my order with order id:-6785678974
Agent: You can cancel your order within 3-5 business days by visiting https://example.com/support or calling +1-200-333-4567.
Customer: how can i get support
Agent: It's fantastic! I understand that you are seeking information about the process, and I'm here to guide you through the process. To proceed with the cancellation, you can try reaching out to our customer support team directly. They will be able to provide you with the necessary information and guide you through the
Customer: exit
Exiting chat...
